In [7]:
import numpy as np
import pandas as pd

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [9]:
df = pd.read_csv('covid_toy.csv')

In [10]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [11]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [13]:
X_train

,age,gender,fever,cough,city
56,71,Male,NaN,Strong,Kolkata
83,17,Female,104.0,Mild,Kolkata
69,73,Female,103.0,Mild,Delhi
43,22,Female,99.0,Mild,Bangalore
70,68,Female,101.0,Strong,Delhi
...,...,...,...,...,...
31,83,Male,103.0,Mild,Kolkata
41,82,Male,NaN,Mild,Kolkata
90,59,Female,99.0,Strong,Delhi
46,19,Female,101.0,Mild,Mumbai


## 1. Aam Zindagi

In [14]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [22]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [16]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [17]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)
# axis=0 (Rows): Stacks arrays vertically on top of each other (increases the number of rows).
# axis=1 (Columns): Aligns arrays next to each other (increases the number of columns, assuming they have the exact same number of rows)
X_train_transformed.shape

(80, 7)

## Mentos Zindagi

In [19]:
from sklearn.compose import ColumnTransformer

In [35]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [43]:
transformer.named_transformers_['tnf2']

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[['Mild', 'Strong']]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide `.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequent categories. If there are infrequentcategories, `max_categories` includes the category representing theinfrequent categories along with the frequent categories. If `None`,there is no limit to the number of output features.`max_categories` do **not** take into account missing or unknowncategories. Setting `unknown_value` or `encoded_missing_value` to aninteger will increase the number of unique integer codes by one each.This can result in up to `max_categories + 2` integer codes... versionadded:: 1.3 Read more in the :ref:`User Guide `.",None


In [39]:
X_transformed = pd.DataFrame(transformer.fit_transform(X_train), columns=transformer.get_feature_names_out())
X_transformed.shape

(80, 7)

In [40]:
transformer.get_feature_names_out()

array(['tnf1__fever', 'tnf2__cough', 'tnf3__gender_Male',
       'tnf3__city_Delhi', 'tnf3__city_Kolkata', 'tnf3__city_Mumbai',
       'remainder__age'], dtype=object)

In [ ]:
enc = transformer.transform(X_test)

# print(enc.get_feature_names_out()) only with fit

AttributeError: 'numpy.ndarray' object has no attribute 'get_feature_names_out'

In [38]:
X_transformed

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,100.794521,1.0,1.0,0.0,1.0,0.0,71.0
1,104.000000,0.0,0.0,0.0,1.0,0.0,17.0
2,103.000000,0.0,0.0,1.0,0.0,0.0,73.0
3,99.000000,0.0,0.0,0.0,0.0,0.0,22.0
4,101.000000,1.0,0.0,1.0,0.0,0.0,68.0
...,...,...,...,...,...,...,...
75,103.000000,0.0,1.0,0.0,1.0,0.0,83.0
76,100.794521,0.0,1.0,0.0,1.0,0.0,82.0
77,99.000000,1.0,0.0,1.0,0.0,0.0,59.0
78,101.000000,0.0,0.0,0.0,0.0,1.0,19.0
